In [1]:
import matplotlib.pyplot as plt
import numpy as np 
from qutip import *

In [ ]:
from scipy.optimize import curve_fit
from qutip import Options
import pandas as pd
opts = {
    "method": "bdf",
    "nsteps": 1e9,
    "atol": 1e-10,
    "rtol": 1e-8,
    "progress_bar": "text",
    'store_final_state': True
}

## Functions for allignment ##
def Ballignment(B, Theta, Phi): 
    return B * np.array([np.sin(Theta) * np.cos(Phi), np.sin(Theta) * np.sin(Phi), np.cos(Theta)])

def beff(g, B, Theta, Phi): 
    B_vec = Ballignment(B, Theta, Phi)
    return B_vec.T @ g

def orthonormal_b(n):
    n = np.asarray(n, dtype=float)
    n /= np.linalg.norm(n)

    # Pick an axis that is NOT parallel to n
    if abs(np.dot(n, [0., 0., 1.])) < 0.9:
        ref = np.array([0., 0., 1.])
    else:
        ref = np.array([1., 0., 0.])

    e1 = ref - np.dot(ref, n) * n
    e1 /= np.linalg.norm(e1)

    e2 = np.cross(n, e1)
    e2 /= np.linalg.norm(e2)

    return e1, e2

## Constants ## 
g_tensor = np.array([[8.5,8.1,0], [8.1,8.5,0], [0,0,.58]])
drive = 1e7
Omega = 5e3
omega_0 = 14.03e9 * 2 * np.pi
L_tot = 1.09e-11 + 6.93e-10
I = 1e-5
phi_ex = 5.01e-15
eta = .984
B = 9.907e-40
Q = 4.4e5
kappa = omega_0 / Q 
Lambda_mag = 5.53e-22 
beta = (1 - (eta/2) * ((phi_ex**2 + Lambda_mag**2) / (I * L_tot)**2) - eta * (-phi_ex/ (I*L_tot)))
# Lambda_mag = 5.53e-22 / (I * L_tot)
Lambda_vec = Lambda_mag  * np.array([0., 0., 1.])
kb = 1.380649e-23
#g = 16.6 
hbar = 1.054e-34
T = 10e-3
nth = 1 / (np.exp( (hbar * omega_0) / (kb * T)) - 1)
muB = 9.274e-24
epsilon_b = hbar * muB / 2 

## Calculating our couplings and such ##

chi = eta * ( (-2 * phi_ex) / (2 * (I*L_tot)**2) + 1/(I*L_tot)) 

## Linear algebra stuff ##
b_vec = beff(g_tensor, B, 0, 0)
b_mag = np.linalg.norm(b_vec)
n_hat = b_vec / b_mag 


## Setting up Hilbert spaces ##
N = 6
a = tensor(destroy(N), qeye(2), qeye(N))
sig_z = tensor(qeye(N), sigmaz(), qeye(N))
sig_x = tensor(qeye(N), sigmax(), qeye(N))
sig_p = tensor(qeye(N), sigmap(), qeye(N))
sig_m = tensor(qeye(N), sigmam(), qeye(N))

b = tensor(qeye(N), qeye(2), destroy(N))
## Drive Sweeps ##
Omega = 2 * np.pi * 4e4 
drive = 1e7
## Set our detuning ##
#Delta = 2 * np.pi * 250.0250e3
delta = 2*np.pi*250e3

Delta = - np.sqrt(delta**2 + Omega**2) - 5
#omega_r = omega_0 * beta - Delta
epsilon_r = 2 * np.pi * drive

alpha = -epsilon_r / (Delta + omega_0 * chi * Lambda_mag - 1j*kappa/2) 
Theta = omega_0 * chi  * np.abs(alpha)**2 * Lambda_vec + (epsilon_b/hbar**2) * b_mag * n_hat 
Theta_mag = np.linalg.norm(Theta) 
n_theta = Theta / Theta_mag
xhat = np.array([1, 0, 0])  
e1, e2 = orthonormal_b(n_theta)


## Getting our spin vectors in our basis ##
sig_vec = np.array([sigmax(), sigmay(), sigmaz()])
sig_1 = np.dot(sig_vec, e1)
sig_2 = np.dot(sig_vec, e2)
sig_theta = np.dot(sig_vec, n_theta)
sig1 = tensor(qeye(N), sig_1, qeye(N))
sig2 = tensor(qeye(N), sig_2, qeye(N))
sigtheta = tensor(qeye(N), sig_theta, qeye(N)) 
sigp = (sig1 + 1j * sig2) / 2
sigm = (sig1 - 1j * sig2) / 2
x1 = np.dot(xhat, e1)
x2 = np.dot(xhat, e2)
xparallel = np.dot(xhat, n_theta)
x_p = x1 - 1j * x2 
x_m = x1 + 1j * x2 
omega_q_ti = 2 * Theta_mag



## Small difference between delta and Delta ##
delta = 2*np.pi*250e3
omega_s = omega_q_ti - delta 
delta_tilde = delta + (Omega**2)/(2*delta) * x_p * x_m 

omega_r = omega_0 * beta + delta_tilde 

n = np.abs(alpha)**2
geff = omega_0 * chi * np.abs(alpha)
g_b = -(geff * Omega * Lambda_mag) / 2 * (1/delta - 1/Delta )


## Full Hamiltonian ##
H0 = Delta * a.dag() * a + (( delta) / 2) * sigtheta
Hint1 =  geff * Lambda_mag * (a.dag() + a)  * sigtheta
Hint2 = (( Omega)/2) * (x_p * sigp + x_m * sigm)


## BS Hamiltonian ##
sigx_spin = (
    xparallel * sig_theta
    + x_p * (sig_1 + 1j*sig_2)/2
    + x_m * (sig_1 - 1j*sig_2)/2
)

H_spin = (
(delta / 2) * sig_theta
+ (Omega / 2) * sigx_spin
)

# Exact dressed basis
spin_evals, spin_evecs = H_spin.eigenstates()

spin_g = spin_evecs[0]
spin_e = spin_evecs[1]

# ACTUAL dressed splitting
omega_d = np.real(spin_evals[1] - spin_evals[0])
spin_evals, spin_evecs = H_spin.eigenstates()
spin_g = spin_evecs[0]
spin_e = spin_evecs[1]

P_g = tensor(
qeye(N),
spin_g * spin_g.dag(), 
qeye(N)
)

P_e = tensor(
    qeye(N),
    spin_e * spin_e.dag(), 
    qeye(N)
)

tau_z = P_e - P_g
tau_p = tensor(
    qeye(N),
    spin_e * spin_g.dag(),
    qeye(N)
)
tau_m = tensor(
    qeye(N),
    spin_g * spin_e.dag(), 
    qeye(N)
)

n_cavity = a.dag() * a
n_dressed_spin = P_e
n_dressed_spin_g = P_g 


g_mag = np.abs(g_b * x_m)                     # coupling strength magnitude
Delta_eff = Delta - delta_tilde                 # effective detuning (delta_tilde already computed above)
Omega_R = np.sqrt(4 * g_mag**2 + Delta_eff**2)   # generalized Rabi frequency
T_R = 2 * np.pi / Omega_R
n_periods = 3          # how many full cycles you want to see
pts_per_period = 200   # resolution per cycle
t = np.linspace(0, 63.5e-6 , 1000)


rho0 = tensor(thermal_dm(N, nth), ket2dm(spin_g), thermal_dm(N, nth))
H_rot = (H0 + Hint1 + Hint2) 

a_t = destroy(N)

rho_test = rho0.ptrace([0])
g2_0 = expect(a_t.dag()**2 * a_t**2, rho_test) / expect(a_t.dag() * a_t, rho_test)**2

print(g2_0)

## Disipation ##
T1 = .44 
T2 = 48e-6
gamma_1 = 1/T1 
gamma_phi = (1/T2) - gamma_1 / 2
dis1_a = np.sqrt(kappa * (nth + 1)) * (a - geff/Delta * sigtheta)
dis1_b = np.sqrt(kappa * nth) * (a.dag() - geff / Delta * sigtheta.dag())
dis2_a = np.sqrt(gamma_1 * (nth + 1)) * (sigm + Omega/(2*delta) * sigtheta + (2*geff)/Delta * (a - a.dag())*sigm) 
dis2_b = np.sqrt(gamma_1 * nth) * (sigp + Omega/(2*delta) * sigtheta.dag() + (2*geff)/Delta * (a.dag() - a)*sigp)
dis3 = np.sqrt(gamma_phi / 2 )* (sigtheta - Omega/delta * tensor(qeye(N), sigx_spin, qeye(N)))

eps = geff * Lambda_mag / Delta    # ≈ -0.049, dimensionless
dis1_a = np.sqrt(kappa*(nth+1)) * (a - eps*sigtheta)
dis1_b = np.sqrt(kappa*nth) * (a.dag() - eps*sigtheta.dag())
dis2_a = np.sqrt(gamma_1*(nth+1)) * (sigm + Omega/(2*delta)*sigtheta + 2*eps*(a - a.dag())*sigm)
dis2_b = np.sqrt(gamma_1*nth) * (sigp + Omega/(2*delta)*sigtheta.dag() + 2*eps*(a.dag() - a)*sigp)

H0 = Delta * a.dag() * a + (( delta_tilde) / 2) * tau_z
H1 =  - (geff**2 * Lambda_mag) / Delta 
H2_tms = g_b * (x_p * a.dag() * tau_p + x_m * a * tau_m)   


tms_result = mesolve(
        H2_tms,
        rho0,
        t,
        e_ops=[n_dressed_spin, n_dressed_spin_g, n_cavity],   
        c_ops = [dis1_a, dis1_b, dis2_a, dis2_b, dis3],
        options=opts, 
        
    )
final_state = tms_result.final_state
rho = final_state 
theta = np.pi / 10
c = tensor(qeye([N,2,N]), destroy(N))
a4 = tensor(a, qeye(N))
vaccuum = basis(N,0)
vac_den = vaccuum * vaccuum.dag()
beamsplitter_density = tensor(rho, vac_den)

U = (1j * theta *(a4.dag() * c + a4 * c.dag())).expm()
rho_def = U.dag() * beamsplitter_density * U 
rho_deffective = rho_def.ptrace([0,1,2])


ket = lambda s, n: tensor(s, basis(N, n))
U_iswap = ket(spin_g, 0) * ket(spin_g, 0).dag()
for k in range(N - 1):
    U_iswap += 1j * ket(spin_g, k + 1) * ket(spin_e, k).dag()
    U_iswap += 1j * ket(spin_e, k) * ket(spin_g, k + 1).dag()
U_iswap += ket(spin_e, N - 1) * ket(spin_e, N - 1).dag()

O = tensor(qeye(N), U_iswap)
sigma = O * rho_deffective * O.dag()
rho = sigma.ptrace([0,2])

## b is our otpical mode a is our microwave here
a2 = tensor(destroy(N), qeye(N))   # microwave
b2 = tensor(qeye(N), destroy(N))   # optical

## assuming the dark clicks happen in a possion distribution:
T_opt = 142e-6                     # https://arxiv.org/pdf/2108.05120
lambda_ex = 5 * T_opt
pdc = 1 - np.exp(- lambda_ex)
rho_click = (b2 * rho * b2.dag()).ptrace(0)     #Keeping microwave 

rho_noclick = rho.ptrace(0)                     #Keeping microwave 


sigma_dm = (1-pdc) * rho_click + pdc * rho_noclick


a1 = destroy(N)
kappa_opt = 1 / T_opt
c_ops_2 = [
    np.sqrt(kappa * (nth + 1)) * a2,
    np.sqrt(kappa * nth) * a2.dag(),
    np.sqrt(kappa_opt) * b2,
]
c_ops_mw = [
    np.sqrt(kappa * (nth + 1)) * a1,
    np.sqrt(kappa * nth) * a1.dag(),
]
c_ops_opt = [np.sqrt(kappa_opt) * a1]
H_free = 0 * a1.dag() * a1   
taulist = np.linspace(0, 1 / kappa, 200)

rho_click_bb   = (b2 * rho * b2.dag()).ptrace(1)   
rho_noclick_bb = rho.ptrace(1)                           
rho_prime_bb   = (1 - pdc) * rho_click_bb + pdc * rho_noclick_bb #optical 
G_bb = np.real(((1-pdc) * rho_click_bb + pdc * rho_noclick_bb2).tr())


rho_click_bb2   = a1 * rho_prime_bb * a1.dag()
rho_noclick_bb2 = rho_prime_bb
G_bb2 = np.real(((1 - pdc) * rho_click_bb2 + pdc * rho_noclick_bb2).tr())

H_free2 = 0 * a2.dag() * a2

I_ab  = tensor(qeye(N), qeye(N))

G_click = np.real(correlation_3op_1t(H_free2, rho, taulist, c_ops_2,
                                     b2.dag(), a2.dag() * a2, b2 ))

G_noclick = np.real(correlation_3op_1t(H_free2, rho, taulist, c_ops_2,
                                       I_ab, a2.dag() * a2, I_ab))

G_ab = (1 - pdc) * G_click + pdc * G_noclick

res_aa = mesolve(H_free2, rho, taulist, c_ops=c_ops_2,
                 e_ops=[I_ab * a2.dag() * a2 * I_ab])
G_aa = np.real(res_aa.expect[0])

# G2_ab = correlation_3op_1t(H_free2, rho, taulist, c_ops_2,
#                            a2.dag(), b2.dag() * b2, a2)


boolean_cs = (G_ab < np.sqrt(G_aa[0] * G_bb))

print(boolean_cs)
print(G_bb)
print(G_aa)
print(G_ab)


2.0000000000000013
10.0%. Run time:   0.28s. Est. time left: 00:00:00:02
20.0%. Run time:   0.35s. Est. time left: 00:00:00:01
30.0%. Run time:   0.52s. Est. time left: 00:00:00:01


40.0%. Run time:   0.59s. Est. time left: 00:00:00:00
50.1%. Run time:   0.63s. Est. time left: 00:00:00:00
60.1%. Run time:   0.69s. Est. time left: 00:00:00:00
70.1%. Run time:   0.75s. Est. time left: 00:00:00:00
80.1%. Run time:   0.81s. Est. time left: 00:00:00:00
90.1%. Run time:   0.85s. Est. time left: 00:00:00:00
100.0%. Run time:   0.89s. Est. time left: 00:00:00:00
Total run time:   0.90s
[False False False False False False False False False False False False
 False False False False False False False False False False False False
 False False False False False False False False False False False False
 False False False False False False False False False False False False
 False False False False False False False False False False False False
 False False False False False False False False False False False False
 False False False False False False False False False False False False
 False False False False False False False False False False False False
 False False 

In [7]:
final_state = tms_result.final_state
rho = final_state 
thetas = np.linspace(0, np.pi/2, 10)
booleans = []

c = tensor(qeye([N,2,N]), destroy(N))
a4 = tensor(a, qeye(N))
vaccuum = basis(N,0)
vac_den = vaccuum * vaccuum.dag()
beamsplitter_density = tensor(rho, vac_den)
for theta in thetas:
    U = (1j * theta *(a4.dag() * c + a4 * c.dag())).expm()
    rho_def = U.dag() * beamsplitter_density * U 
    rho_deffective = rho_def.ptrace([0,1,2])


    ket = lambda s, n: tensor(s, basis(N, n))
    U_iswap = ket(spin_g, 0) * ket(spin_g, 0).dag()
    for k in range(N - 1):
        U_iswap += 1j * ket(spin_g, k + 1) * ket(spin_e, k).dag()
        U_iswap += 1j * ket(spin_e, k) * ket(spin_g, k + 1).dag()
    U_iswap += ket(spin_e, N - 1) * ket(spin_e, N - 1).dag()

    O = tensor(qeye(N), U_iswap)
    sigma = O * rho_deffective * O.dag()
    rho = sigma.ptrace([0,2])

    ## b is our otpical mode a is our microwave here
    a2 = tensor(destroy(N), qeye(N))   # microwave
    b2 = tensor(qeye(N), destroy(N))   # optical

    ## assuming the dark clicks happen in a possion distribution:
    T_opt = 142e-6                      # https://arxiv.org/pdf/2108.05120 site a lifetime in silicon 
    lambda_ex = 5 * T_opt
    pdc = 1 - np.exp(- lambda_ex)
    rho_click = (b2 * rho * b2.dag()).ptrace(0)     #Keeping microwave 

    rho_noclick = rho.ptrace(0)                     #Keeping microwave 


    sigma_dm = (1-pdc) * rho_click + pdc * rho_noclick


    a1 = destroy(N)
    kappa_opt = 1 / T_opt
    c_ops_2 = [
        np.sqrt(kappa * (nth + 1)) * a2,
        np.sqrt(kappa * nth) * a2.dag(),
        np.sqrt(kappa_opt) * b2,
    ]
    c_ops_mw = [
        np.sqrt(kappa * (nth + 1)) * a1,
        np.sqrt(kappa * nth) * a1.dag(),
    ]
    c_ops_opt = [np.sqrt(kappa_opt) * a1]
    H_free = 0 * a1.dag() * a1   
    taulist = np.linspace(0, 1 / kappa, 200)

    rho_click_bb   = (b2 * rho * b2.dag()).ptrace(1)   
    rho_noclick_bb = rho.ptrace(1)                           
    rho_prime_bb   = (1 - pdc) * rho_click_bb + pdc * rho_noclick_bb #optical 

    rho_click_bb2   = a1 * rho_prime_bb * a1.dag()
    rho_noclick_bb2 = rho_prime_bb
    G_bb = np.real(((1 - pdc) * rho_click_bb2 + pdc * rho_noclick_bb2).tr())

    H_free2 = 0 * a2.dag() * a2

    I_ab  = tensor(qeye(N), qeye(N))

    G_click = np.real(correlation_3op_1t(H_free2, rho, taulist, c_ops_2,
                                        b2.dag(), a2.dag() * a2, b2 ))

    G_noclick = np.real(correlation_3op_1t(H_free2, rho, taulist, c_ops_2,
                                        I_ab, a2.dag() * a2, I_ab))

    G_ab = (1 - pdc) * G_click + pdc * G_noclick

    # G_aa = np.real(correlation_3op_1t(H_free2, rho, taulist, c_ops_2, a2.dag(), a2.dag() * a2, a2))
    res_aa = mesolve(H_free2, rho, taulist, c_ops=c_ops_2,
                 e_ops=[a2.dag() * a2.dag() * a2 * a2])
    G_aa = np.real(res_aa.expect[0])

    # G2_ab = correlation_3op_1t(H_free2, rho, taulist, c_ops_2,
    #                            a2.dag(), b2.dag() * b2, a2)


    boolean_cs = (G_ab < np.sqrt(G_aa[0] * G_bb))

    print(boolean_cs)
    booleans.append(boolean_cs[0])
    print(G_aa[0])

print(thetas)
print(booleans)

[False False False False False False False False False False False False
 False False False False False False False False False False False False
 False False False False False False False False False False False False
 False False False False False False False False False False False False
 False False False False False False False False False False False False
 False False False False False False False False False False False False
 False False False False False False False False False False False False
 False False False False False False False False False False False False
 False False False False False False False False False False False False
 False False False False False False False False False False False False
 False False False False False False False False False False False False
 False False False False False False False False False False False False
 False False False False False False False False False False False False
 False False False False False False False False Fa